# Day 17: Multiple Tools

Yesterday we gave the LLM one tool. Today we give it **multiple tools** and let it choose.

The LLM becomes a router — deciding which tool to use based on the question.

## Setup

In [45]:
from google import genai
from google.genai import types
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path='../.env')
API_KEY = os.environ["GEMINI_API_KEY"]
client = genai.Client(api_key=API_KEY)

## Define Multiple Tools

Let's create a travel assistant with 3 tools:
1. **get_ticket_price** — Get flight prices
2. **get_weather** — Get weather for a city
3. **book_flight** — Book a flight

In [46]:
# Tool 1: Ticket prices
ticket_prices = {
    "london": 799,
    "paris": 899,
    "tokyo": 1400,
    "berlin": 499,
    "new york": 599
}

def get_ticket_price(destination_city: str) -> str:
    """Get the price of a ticket to a destination city."""
    print(f"🎫 Tool: get_ticket_price({destination_city})")
    city = destination_city.lower()
    price = ticket_prices.get(city, None)
    if price:
        return f"Flight to {destination_city}: ${price}"
    return f"No flights available to {destination_city}"

In [47]:
# Tool 2: Weather
weather_data = {
    "london": {"temp": 15, "condition": "Cloudy"},
    "paris": {"temp": 18, "condition": "Sunny"},
    "tokyo": {"temp": 22, "condition": "Clear"},
    "berlin": {"temp": 12, "condition": "Rainy"},
    "new york": {"temp": 20, "condition": "Partly Cloudy"}
}

def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    print(f"🌤️ Tool: get_weather({city})")
    city_lower = city.lower()
    weather = weather_data.get(city_lower, None)
    if weather:
        return f"Weather in {city}: {weather['temp']}°C, {weather['condition']}"
    return f"Weather data not available for {city}"

In [48]:
# Tool 3: Book flight
def book_flight(destination_city: str, passenger_name: str) -> str:
    """Book a flight for a passenger."""
    print(f"✈️ Tool: book_flight({destination_city}, {passenger_name})")
    city = destination_city.lower()
    price = ticket_prices.get(city, None)
    if price:
        confirmation = f"BK{hash(passenger_name + city) % 10000:04d}"
        return f"✅ Booked! {passenger_name} → {destination_city}. Confirmation: {confirmation}. Total: ${price}"
    return f"Cannot book flight to {destination_city} - destination not available"

## Define Tool Schemas

Each tool needs a schema so the LLM knows what it does.

In [49]:
# All tools in one Tool object
travel_tools = types.Tool(
    function_declarations=[
        # Tool 1: Ticket price
        types.FunctionDeclaration(
            name="get_ticket_price",
            description="Get the price of a flight ticket to a destination city.",
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "destination_city": types.Schema(
                        type=types.Type.STRING,
                        description="The city to fly to"
                    )
                },
                required=["destination_city"]
            )
        ),
        # Tool 2: Weather
        types.FunctionDeclaration(
            name="get_weather",
            description="Get the current weather for a city.",
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "city": types.Schema(
                        type=types.Type.STRING,
                        description="The city to check weather for"
                    )
                },
                required=["city"]
            )
        ),
        # Tool 3: Book flight
        types.FunctionDeclaration(
            name="book_flight",
            description="Book a flight ticket for a passenger to a destination.",
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "destination_city": types.Schema(
                        type=types.Type.STRING,
                        description="The city to fly to"
                    ),
                    "passenger_name": types.Schema(
                        type=types.Type.STRING,
                        description="Name of the passenger"
                    )
                },
                required=["destination_city", "passenger_name"]
            )
        )
    ]
)

print("✅ Defined 3 tools: get_ticket_price, get_weather, book_flight")

✅ Defined 3 tools: get_ticket_price, get_weather, book_flight


## Tool Dispatcher

A function that routes to the correct tool based on what the LLM requests.

In [50]:
def execute_tool(func_call):
    """Execute the tool requested by the LLM."""
    name = func_call.name
    args = func_call.args
    
    if name == "get_ticket_price":
        return get_ticket_price(args["destination_city"])
    elif name == "get_weather":
        return get_weather(args["city"])
    elif name == "book_flight":
        return book_flight(args["destination_city"], args["passenger_name"])
    else:
        return f"Unknown tool: {name}"

## The Multi-Tool Chat Function

In [51]:
def chat(message):
    """Chat with multiple tools available."""
    
    # First call - LLM decides which tool (if any) to use
    response = client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=message,
        config=types.GenerateContentConfig(
            tools=[travel_tools]
        )
    )
    
    part = response.candidates[0].content.parts[0]
    
    # Check if LLM wants to call a tool
    if part.function_call:
        func_call = part.function_call
        print(f"\n🔧 LLM chose tool: {func_call.name}")
        
        # Execute the tool
        tool_result = execute_tool(func_call)
        print(f"📋 Result: {tool_result}")
        
        # Send result back for final answer
        follow_up = client.models.generate_content(
            model="gemini-2.5-flash-lite",
            contents=[
                types.Content(role="user", parts=[types.Part.from_text(text=message)]),
                types.Content(role="model", parts=[part]),
                types.Content(role="user", parts=[
                    types.Part.from_function_response(
                        name=func_call.name,
                        response={"result": tool_result}
                    )
                ])
            ],
            config=types.GenerateContentConfig(
                tools=[travel_tools]
            )
        )
        return follow_up.text
    
    # No tool needed
    return response.text

## Test: LLM Chooses the Right Tool

In [52]:
# Test 1: Should use get_ticket_price
print("❓ How much is a flight to Paris?")
print(f"\n💬 {chat('How much is a flight to Paris?')}")

❓ How much is a flight to Paris?

🔧 LLM chose tool: get_ticket_price
🎫 Tool: get_ticket_price(Paris)
📋 Result: Flight to Paris: $899

💬 The flight to Paris is $899.


In [53]:
# Test 2: Should use get_weather
print("❓ What's the weather like in Tokyo?")
print(f"\n💬 {chat('What is the weather like in Tokyo?')}")

❓ What's the weather like in Tokyo?

🔧 LLM chose tool: get_weather
🌤️ Tool: get_weather(Tokyo)
📋 Result: Weather in Tokyo: 22°C, Clear

💬 The weather in Tokyo is 22°C and clear.


In [54]:
# Test 3: Should use book_flight
print("❓ Book a flight to London for John Smith")
print(f"\n💬 {chat('Book a flight to London for John Smith')}")

❓ Book a flight to London for John Smith

🔧 LLM chose tool: book_flight
✈️ Tool: book_flight(London, John Smith)
📋 Result: ✅ Booked! John Smith → London. Confirmation: BK1830. Total: $799


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 2.707003579s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flas

In [ ]:
# Test 4: No tool needed
print("❓ What airlines do you work with?")
print(f"\n💬 {chat('What airlines do you work with?')}")

## Key Takeaways

1. **LLM as a router** — It reads the question and picks the right tool
2. **Clear descriptions matter** — The better you describe each tool, the better the LLM chooses
3. **Same pattern** — Request → Tool Call → Execute → Response
4. **Scalable** — Add more tools, same flow works

---

**Next:** Day 18 — Agent Loop (multiple tool calls in sequence)